<a href="https://colab.research.google.com/github/cksingh158-code/Differential_physics/blob/main/Differentiable_Cylinder_Flow_to_Find_Critical_Re.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install phiflow
!pip install torch
!pip install numpy
!pip install matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.4/207.4 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.0/373.0 kB 18.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for phiflow: filename=phiflow-3.4.0-py3-none-any.whl size=238958 sha256=1ffd2e449a0bc35d98824eb3c623835fd4e6b7bff0559bbe46a47b57626b5533
  Stored in directory: /root/.cache/pip/wheels/e1/01/57/99aaeb0297ecc2d43bab373e905e5d73be8745183fc5bc47e7
  Created wheel for phiml: filename=phiml-1.15.1-py3-none-any.whl size=390712 sha256=9a4d8870ee1e0edd451842b8248a86e31e2270f2b27b0dd0a22a7ad710a606ad
  Stored in directory: /root/.cache/pip/wheels/e1/d7/22/bdfc5b004ef0a7c96f3a201c315d3e9ef5d5b714a3b48e9fd9
Successfully built phiflow phiml


In [ ]:
# Differentiable Cylinder Flow to Find Critical Re
import torch
import numpy as np
from phi.torch.flow import *

# ----------------------------
# 1. Simulation Parameters
# ----------------------------
Nx, Ny = 64, 32          # grid resolution
domain = Box(x=(0, 4), y=(0, 2))
U = 1.0                   # inlet velocity
D = 1.0                   # cylinder diameter
probe_point = (1.5, 1.0)  # behind cylinder

# ----------------------------
# 2. Differentiable Viscosity
# ----------------------------
# Start with low Re (high viscosity)
nu = torch.tensor(0.01, requires_grad=True, dtype=torch.float32)

# Optimizer
optimizer = torch.optim.Adam([nu], lr=1e-3)

# Cylinder obstacle
cylinder = Sphere(x=1.0, y=1.0, radius=D/2)

# ----------------------------
# 3. Optimization Loop
# ----------------------------
num_iters = 30          # number of optimization iterations
sim_steps = 100          # time steps per iteration
dt = 0.05

for iter in range(num_iters):
    # Initialize velocity field directly with the uniform flow
    velocity = StaggeredGrid((U, 0), extrapolation.BOUNDARY, x=Nx, y=Ny, bounds=domain)

    probe_values = []

    # Run simulation
    for step in range(sim_steps):
        # Semi-Lagrangian advection
        velocity = advect.semi_lagrangian(velocity, velocity, dt=dt)

        # Explicit diffusion (viscosity)
        velocity = diffuse.explicit(velocity, nu, dt=dt)

        # Enforce incompressibility
        velocity, _ = fluid.make_incompressible(velocity)

        # Apply obstacle (smoothed for differentiability)
        velocity = fluid.apply_boundary_conditions(velocity, obstacles=[Obstacle(cylinder)])

        # Record probe velocity behind cylinder
        probe_values.append(velocity.at(Point(math.vec(x=probe_point[0], y=probe_point[1])))[0].values.native())

    # Convert probe series to torch tensor
    probe_tensor = torch.stack(probe_values)

    # Loss: negative variance to maximize unsteadiness
    loss = -probe_tensor.var()

    # Gradient step
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Clamp viscosity to avoid negative
    with torch.no_grad():
        nu.clamp_(0.001, 0.1)

    # Compute Re for reporting
    Re = U * D / nu.item()

    print(f"Iter {iter+1}/{num_iters} | nu={nu.item():.5f} | Re={Re:.2f} | Loss={loss.item():.6f}")

Iter 1/30 | nu=0.01100 | Re=90.92 | Loss=-0.000006
Iter 2/30 | nu=0.01200 | Re=83.35 | Loss=-0.000006
Iter 3/30 | nu=0.01299 | Re=76.97 | Loss=-0.000006
Iter 4/30 | nu=0.01399 | Re=71.50 | Loss=-0.000006
Iter 5/30 | nu=0.01498 | Re=66.77 | Loss=-0.000006
Iter 6/30 | nu=0.01596 | Re=62.64 | Loss=-0.000006
Iter 7/30 | nu=0.01695 | Re=59.01 | Loss=-0.000006
Iter 8/30 | nu=0.01793 | Re=55.78 | Loss=-0.000006
Iter 9/30 | nu=0.01890 | Re=52.90 | Loss=-0.000006
Iter 10/30 | nu=0.01988 | Re=50.31 | Loss=-0.000006
Iter 11/30 | nu=0.02085 | Re=47.97 | Loss=-0.000006
Iter 12/30 | nu=0.02181 | Re=45.86 | Loss=-0.000006
Iter 13/30 | nu=0.02276 | Re=43.94 | Loss=-0.000006
Iter 14/30 | nu=0.02365 | Re=42.28 | Loss=-0.000006
Iter 15/30 | nu=0.02452 | Re=40.79 | Loss=-0.000006
Iter 16/30 | nu=0.02544 | Re=39.31 | Loss=-0.000006
Iter 17/30 | nu=0.02640 | Re=37.87 | Loss=-0.000006
Iter 18/30 | nu=0.02739 | Re=36.51 | Loss=-0.000006
Iter 19/30 | nu=0.02840 | Re=35.21 | Loss=-0.000006
Iter 20/30 | nu=0.029

/usr/local/lib/python3.12/dist-packages/phi/physics/diffuse.py:54: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  warnings.warn(f"CFL condition violated (CFL = {float(cfl.max):.1f} > 0.5) in diffuse.explicit() with diffusivity={diffusivity}, dt={dt}, dx={u.dx}. Increase substeps or use diffuse.implicit() instead.", RuntimeWarning, stacklevel=2)
/tmp/ipykernel_413/3042895544.py:46: RuntimeWarning: CFL condition violated (CFL = 0.5 > 0.5) in diffuse.explicit() with diffusivity=0.03932015970349312, dt=0.05, dx=(x=0.062, y=0.062) float64. Increase substeps or use diffuse.implicit() instead.
  velocity = diffuse.explicit(velocity, nu, dt=dt)
/tmp/ipykernel_413/3042895544.py:46: RuntimeWarning: CFL condition violated (CFL = 0.5 > 0.5) in diffuse.explicit() with diffusivity=0.03932015970349312, dt=0.05,

Iter 30/30 | nu=0.03983 | Re=25.11 | Loss=-0.000008
